In [7]:
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import os

In [8]:
load_dotenv(override=True)

expert_file = os.getenv("EXPERT_FILE")
output_folder_path = os.getenv("OUTPUT_FOLDER_PATH")


In [9]:
df = pd.read_excel(expert_file, engine='openpyxl')
df

,N,Imagen,Ingredientes,Peso,kcal,P (g),CH (g),G (g),Observaciones
0,1.0,NaN,Malta,50.0,185.0,1.9,22.5,NaN,NaN
1,NaN,NaN,Agua,300.0,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Lúpulo,3.0,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,Levadura,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
290,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
291,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
292,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
def clean_aggregate_data(df, output_path=output_folder_path,  filename_base="expert_cleaned"):
    df_clean = df.copy()
    df_clean['N'] = df_clean['N'].ffill()
    df_clean = df_clean.dropna(subset=['N'])

    result = df_clean.groupby('N').agg({
        'Ingredientes': lambda x: ', '.join(x.dropna().astype(str)),
        'kcal': 'sum',
        'P (g)': 'sum',
        'CH (g)': 'sum',
        'G (g)': 'sum',
        'Peso': 'sum',
        'Observaciones': lambda x: ', '.join(x.dropna().astype(str))
    })

    result['Ingredientes'] = result['Ingredientes'].replace('', np.nan)

    result = result.reset_index()

    result = result.rename(columns={
        'N': 'id',
        'Ingredientes': 'description',
        'kcal': 'calories',
        'P (g)': 'proteins',
        'CH (g)': 'carbohydrates',
        'G (g)': 'fats',
        'Peso': 'serving_size',
        'Observaciones': 'observaciones'
    })

    # Get current project directory
    current_dir = os.getcwd()

    # Define output file paths
    excel_path = os.path.join(output_path, f"{filename_base}.xlsx")
    json_path = os.path.join(current_dir, f"{filename_base}.json")

    # Export files
    result.to_excel(excel_path, index=False)
    result.to_json(json_path, orient='records', indent=2, force_ascii=False)

    print(f"Data exported to:\n- Excel: {excel_path}\n- JSON: {json_path}")
    return result



In [11]:
clean_aggregate_data(df)

Data exported to:
- Excel: /Users/martinhachiya/dev/datasets/nutria_backbone/llm_response/expert_cleaned.xlsx
- JSON: /Users/martinhachiya/dev/nutria_backbone/nutria_solution/expert/expert_cleaned.json


,id,description,calories,proteins,carbohydrates,fats,serving_size,observaciones
0,1.0,"Malta, Agua, Lúpulo, Levadura",185.00,1.900,22.50,0.000,353.0,
1,2.0,"Aceituna, Ajo, Pasta, Oregano, Aceite de oliva...",274.75,4.990,49.10,7.800,268.0,
2,3.0,"Tomate, Salmón, Huevo, Croasant",514.00,32.400,33.00,27.400,340.0,
3,4.0,"Papas fritas, Aceite de oliva, Carne de res, S...",547.00,24.300,65.00,31.000,223.0,
4,5.0,"Remolacha, Arroz, Carne de res",371.50,25.300,40.30,12.050,340.0,
5,6.0,"Brocoli, Lechuga, Carne de res, Tomate",206.75,21.675,4.55,11.425,205.0,
6,7.0,"Palta, Salmón",223.00,20.700,2.60,14.300,160.0,
7,8.0,"Café espresso, Agua",3.00,0.000,0.50,0.100,30.0,
8,9.0,"Café, Leche",58.00,3.950,6.40,1.800,150.0,
9,10.0,Agua,0.00,0.000,0.00,0.000,500.0,
